# Lab 22 — DPO/ORPO Alignment (T4 tier · Part 1)

**Track 3 · Day 22 · VinUni AICB program**

This Colab notebook runs the long-training half of the lab:
1. SFT-mini build (NB1)
2. Preference data prep (NB2)
3. DPO training (NB3)

At the end, it packages the handoff artifacts into `day22-colab-part1.zip` so you can
download them and continue in Part 2 with a fresh Colab runtime.


## A. Colab setup — install deps + set tier
(Skip these cells if running in Jupyter from the lab repo.)

In [ ]:
# Set tier early — every downstream cell reads this.
import os
os.environ["COMPUTE_TIER"] = "T4"
print(f"COMPUTE_TIER set to {os.environ['COMPUTE_TIER']}")

In [ ]:
# Install required packages (~3-5 min on Colab)
!pip install -q unsloth>=2025.10 trl>=0.12 peft>=0.13 bitsandbytes>=0.44 \
                datasets>=3.1 accelerate>=1.1 llama-cpp-python>=0.3 \
                "lm-eval[ifeval,math]>=0.4.5" \
                matplotlib pandas pyarrow openai anthropic

In [ ]:
# Probe GPU
import torch
assert torch.cuda.is_available(), "Enable GPU runtime: Runtime → Change runtime type → GPU"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}  ({gpu.total_memory / 1e9:.1f} GB)")

In [ ]:
# Set up working directory matching the repo layout — Colab runs from /content
from pathlib import Path
WORK = Path("/content/lab22")
WORK.mkdir(exist_ok=True)
(WORK / "notebooks").mkdir(exist_ok=True)
(WORK / "data" / "pref").mkdir(parents=True, exist_ok=True)
(WORK / "data" / "eval").mkdir(parents=True, exist_ok=True)
(WORK / "adapters" / "sft-mini").mkdir(parents=True, exist_ok=True)
(WORK / "adapters" / "dpo").mkdir(parents=True, exist_ok=True)
(WORK / "adapters" / "merged-fp16").mkdir(parents=True, exist_ok=True)
(WORK / "gguf").mkdir(exist_ok=True)
(WORK / "submission" / "screenshots").mkdir(parents=True, exist_ok=True)
import os
os.chdir(WORK / "notebooks")
print(f"Working dir: {Path.cwd()}")

---
## Stages 1-3 in this notebook
Run cells in order. This notebook stops after NB3 and downloads a handoff zip for Part 2.
---


---
# ⏵ Stage from `notebooks/01_sft_mini.py`
---

# NB1 — SFT-mini: Build the Lab 21 SFT checkpoint inline

**Stack:** Unsloth + LoRA r=16 + bitsandbytes 4-bit base + 1k VN Alpaca, 1 epoch.
Maps to deck §1 (why SFT alone insufficient — motivates the upcoming DPO step) +
deck §3 (DPO will need this SFT checkpoint as initial policy).

> **Mục tiêu:** tạo 1 SFT adapter "đủ tốt" để DPO có gì align lên. Đây là
> Lab 21 thu gọn — nếu bạn đã hoàn thành Lab 21 sibling repo
> ([VinUni-AI20k/Day21-Track3-Finetuning-LLMs-LoRA-QLoRA](https://github.com/VinUni-AI20k/Day21-Track3-Finetuning-LLMs-LoRA-QLoRA)),
> bạn có thể SKIP notebook này và copy adapter cũ vào `adapters/sft-mini/`.
>
> Nếu chưa, notebook này build từ đầu trong ~10 phút trên T4 (15 phút trên Colab CPU runtime — *đừng làm vậy*).

## 0. Setup

In [ ]:
import os
from pathlib import Path

# Tier detection. Defaults to T4 if env not set.
COMPUTE_TIER = os.environ.get("COMPUTE_TIER", "T4").upper()
assert COMPUTE_TIER in ("T4", "BIGGPU"), f"Invalid COMPUTE_TIER: {COMPUTE_TIER}"

# Tier-specific hyperparameters
if COMPUTE_TIER == "T4":
    BASE_MODEL = "unsloth/Qwen2.5-3B-bnb-4bit"
    MAX_LEN = 512
    PER_DEVICE_BATCH = 1
    GRAD_ACCUM = 8
else:  # BIGGPU
    BASE_MODEL = "unsloth/Qwen2.5-7B-bnb-4bit"
    MAX_LEN = 1024
    PER_DEVICE_BATCH = 2
    GRAD_ACCUM = 4

SFT_DATASET = os.environ.get("SFT_DATASET", "5CD-AI/Vietnamese-alpaca-gpt4-gg-translated")
SFT_SLICE = 1000
NUM_EPOCHS = 1

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ADAPTER_OUT = REPO_ROOT / "adapters" / "sft-mini"
ADAPTER_OUT.mkdir(parents=True, exist_ok=True)

print(f"COMPUTE_TIER:    {COMPUTE_TIER}")
print(f"BASE_MODEL:      {BASE_MODEL}")
print(f"SFT_DATASET:     {SFT_DATASET}  (slice: {SFT_SLICE})")
print(f"max_seq_length:  {MAX_LEN}")
print(f"effective batch: {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"output:          {ADAPTER_OUT}")

In [ ]:
import torch

assert torch.cuda.is_available(), "DPO needs a CUDA GPU. See HARDWARE-GUIDE.md."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}  ({gpu.total_memory / 1e9:.1f} GB)")

## 1. Load base model with Unsloth

Unsloth bundles patched 4-bit kernels — that's how Qwen2.5-3B (or 7B) stays
in T4 / A100 budget. The `FastLanguageModel.from_pretrained` call returns a
4-bit quantized base; `get_peft_model` attaches the LoRA adapter on top.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_LEN,
    dtype=None,                # auto: bf16 on Ampere+, fp16 on Turing
    load_in_4bit=True,
)

# Critical for batch training — Qwen tokenizers ship without pad token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Set tokenizer.pad_token = eos_token")

CHATML_FALLBACK_TEMPLATE = """{%- for message in messages %}
{{- '<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>
' }}
{%- endfor %}
{%- if add_generation_prompt %}
{{- '<|im_start|>assistant
' }}
{%- endif %}
"""
if not getattr(tokenizer, "chat_template", None):
    tokenizer.chat_template = CHATML_FALLBACK_TEMPLATE
    print("Set fallback tokenizer.chat_template (ChatML)")


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,           # Unsloth supports dropout=0 for free speed
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",  # 30% VRAM savings
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 2. Load + format VN Alpaca slice

`5CD-AI/Vietnamese-alpaca-gpt4-gg-translated` is a ~52k-row VN Alpaca-style
translation. Lab 21 uses a 1k slice for the demo run; we keep that scale so
reward-gap comparisons stay lightweight on Colab.

In [ ]:
from datasets import load_dataset

# Always load SFT split right before formatting to avoid accidental overwrite
# from later cells (preference section also uses variable name `ds`).
ds = load_dataset(SFT_DATASET, split=f"train[:{SFT_SLICE}]")
print(f"Loaded {len(ds)} rows. Columns: {ds.column_names}")
if len(ds) == 0:
    raise ValueError("SFT dataset is empty. Check SFT_DATASET / split.")
print(f"\nFirst row:\n{ds[0]}")

In [ ]:
# Alpaca VI -> ChatML format (Qwen2.5 native template)
def _get_text(row, keys):
    for k in keys:
        if k in row:
            v = row[k]
            if v is not None:
                s = str(v).strip()
                if s != "":
                    return s
    return ""


def format_alpaca_to_chat(row):
    # Prefer Vietnamese fields; fallback to generic field names.
    instruction = _get_text(row, ["instruction_vi", "instruction", "prompt", "question", "instruction_en"])
    input_text = _get_text(row, ["input_vi", "input", "context", "input_en"])
    output_text = _get_text(row, ["output_vi", "output", "response", "answer", "output_en"])

    if not instruction or not output_text:
        return {"text": ""}

    prompt = instruction + (f"\n\n{input_text}" if input_text else "")
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": output_text},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


ds_formatted = ds.map(format_alpaca_to_chat, remove_columns=ds.column_names)
ds_formatted = ds_formatted.filter(lambda x: isinstance(x["text"], str) and len(x["text"].strip()) > 0)

print(f"Rows formatted: {len(ds_formatted)}")
if len(ds_formatted) == 0:
    raise ValueError(
        "No valid formatted rows. Re-run the SFT load cell immediately above this one."
    )
print(f"\nSample formatted text (first 500 chars):\n{ds_formatted[0]['text'][:500]}")

## 3. Train SFT-mini

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=str(ADAPTER_OUT.parent / "sft-mini-checkpoints"),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="no",        # Save only at the end via trainer.model.save_pretrained
    optim="adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    seed=42,
    max_seq_length=MAX_LEN,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=sft_config,
    train_dataset=ds_formatted,
)

In [ ]:
train_result = trainer.train()
print(f"\nFinal train loss: {train_result.training_loss:.4f}")

### 3a. Plot loss curve (deliverable: `02_sft_loss.png`)

In [ ]:
import matplotlib.pyplot as plt

losses = [log["loss"] for log in trainer.state.log_history if "loss" in log]
steps = [log["step"] for log in trainer.state.log_history if "loss" in log]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, losses, marker="o", markersize=3, linewidth=1.2)
ax.set_xlabel("Training step")
ax.set_ylabel("Loss")
ax.set_title(f"SFT-mini loss · {COMPUTE_TIER} · {BASE_MODEL.split('/')[-1]} · {SFT_SLICE} samples")
ax.grid(True, alpha=0.3)
fig.tight_layout()
screenshot_dir = REPO_ROOT / "submission" / "screenshots"
screenshot_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(screenshot_dir / "02-sft-loss.png", dpi=120)
plt.show()

## 4. Save adapter + sanity-check generation

In [ ]:
trainer.model.save_pretrained(str(ADAPTER_OUT))
tokenizer.save_pretrained(str(ADAPTER_OUT))
print(f"Saved SFT adapter to {ADAPTER_OUT}")

In [ ]:
# Sanity: generate 1 sample to confirm the adapter loaded correctly.
FastLanguageModel.for_inference(model)
prompt = "Giải thích ngắn gọn (3-4 câu) thuật toán quicksort hoạt động thế nào."
messages = [{"role": "user", "content": prompt}]
inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True
).to("cuda")
with torch.no_grad():
    out = model.generate(input_ids=inputs, max_new_tokens=200, do_sample=False)
generated = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print(f"PROMPT: {prompt}\n")
print(f"SFT-mini response:\n{generated}")

## 5. Vibe-coding callout

Bạn vừa tái tạo Lab 21 trong ~10 phút. Một câu hỏi để brainstorm:

> **Thật ra, bạn cần *bao nhiêu* SFT để DPO có ý nghĩa?**
>
> Thử thay `SFT_SLICE = 1000` → `100`. Re-run NB1 → NB3 với SFT yếu hơn.
> Quan sát: reward gap có còn tăng được không? Output coherent không?
>
> Đây là 1 design decision *think-hard zone* (xem VIBE-CODING.md): không có
> đáp án sẵn trong deck. Hypothesize trước, train sau, viết kết quả vào
> `submission/REFLECTION.md` § 6.

**Next:** NB2 — load + format preference data.

---
# ⏵ Stage from `notebooks/02_preference_data.py`
---

# NB2 — Preference Data

**Stack:** `argilla/ultrafeedback-binarized-preferences-cleaned` + tokenizer apply_chat_template.
Maps to deck §5.1 (preference data formats) + §5.4 (VN landscape — what exists vs not).

> **Mục tiêu:** load preference dataset, format thành `{prompt, chosen, rejected}` với
> chat template Qwen2.5, lưu Parquet vào `data/pref/`. Không train gì cả — đây là pure
> data prep.
>
> Deck §5.4 lists VN preference data realities:
> - **VinaLLaMA / PhoGPT / Vistral**: SFT-only, no published DPO data.
> - **SeaLLM / Sailor2**: DPO-aligned, Sailor2 has `Sailor2-translated-ultrafeedback-vi`.
> - **Native VN preference**: gap. **Bonus B** (xem `BONUS-CHALLENGE.md`) là cơ hội build.

## 0. Setup

In [ ]:
import os
from pathlib import Path

COMPUTE_TIER = os.environ.get("COMPUTE_TIER", "T4").upper()

if COMPUTE_TIER == "T4":
    PREF_SLICE = 2000
    MAX_LEN = 512
    MAX_PROMPT_LEN = 256
else:
    PREF_SLICE = 5000
    MAX_LEN = 1024
    MAX_PROMPT_LEN = 512

PREF_DATASET = os.environ.get(
    "PREF_DATASET", "argilla/ultrafeedback-binarized-preferences-cleaned"
)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ADAPTER_DIR = REPO_ROOT / "adapters" / "sft-mini"
PREF_OUT = REPO_ROOT / "data" / "pref"
PREF_OUT.mkdir(parents=True, exist_ok=True)

print(f"COMPUTE_TIER:    {COMPUTE_TIER}")
print(f"PREF_DATASET:    {PREF_DATASET}  (slice: {PREF_SLICE})")
print(f"MAX_LEN:         {MAX_LEN}")
print(f"MAX_PROMPT_LEN:  {MAX_PROMPT_LEN}")
print(f"output:          {PREF_OUT}")

## 1. Load tokenizer (matches NB1 base model)

In [ ]:
from transformers import AutoTokenizer

assert ADAPTER_DIR.exists(), f"NB1 must run first — {ADAPTER_DIR} missing"
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer: {tokenizer.__class__.__name__}  vocab={tokenizer.vocab_size:,}")

CHATML_FALLBACK_TEMPLATE = """{%- for message in messages %}
{{- '<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>
' }}
{%- endfor %}
{%- if add_generation_prompt %}
{{- '<|im_start|>assistant
' }}
{%- endif %}
"""
if not getattr(tokenizer, "chat_template", None):
    tokenizer.chat_template = CHATML_FALLBACK_TEMPLATE
    print("Set fallback tokenizer.chat_template (ChatML)")


## 2. Load UltraFeedback (English baseline)

**Why English?** UltraFeedback was the canonical preference dataset of the deck
demo (§7.1: "2k UltraFeedback pairs, 30 min A100, 3.2 → 4.1 helpfulness"). Using
the same dataset = numbers comparable to deck.

**Why not Vietnamese?** Native VN preference data is a gap (deck §5.4). Translated
data (`Sailor2-translated-ultrafeedback-vi`) exists but is NLLB-MT-quality, not native.
Bonus B has the full provocation.

In [ ]:
from datasets import load_dataset

ds = load_dataset(PREF_DATASET, split=f"train[:{PREF_SLICE}]")
print(f"Loaded {len(ds)} pairs. Columns: {ds.column_names}")

## 3. Format with chat template

DPO Trainer expects `prompt / chosen / rejected` columns. Each must already
include the chat template tokens — Trainer doesn't apply template internally.

In [ ]:
def format_pref(row):
    prompt_msgs = [{"role": "user", "content": row["prompt"]}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_msgs, tokenize=False, add_generation_prompt=True
    )
    # `chosen` and `rejected` in this dataset are list-of-dicts with role/content.
    # Take just the assistant turn text (last message).
    chosen_text = row["chosen"][-1]["content"] if isinstance(row["chosen"], list) else row["chosen"]
    rejected_text = row["rejected"][-1]["content"] if isinstance(row["rejected"], list) else row["rejected"]
    return {
        "prompt": prompt_text,
        "chosen": chosen_text,
        "rejected": rejected_text,
    }


pref = ds.map(format_pref, remove_columns=ds.column_names)
print(f"Formatted: {len(pref)} pairs · cols: {pref.column_names}")

### 3a. Inspect 3 examples + token counts (deliverable: NB2 rubric §2)

In [ ]:
import textwrap

for i in range(3):
    row = pref[i]
    n_prompt = len(tokenizer(row["prompt"]).input_ids)
    n_chosen = len(tokenizer(row["chosen"]).input_ids)
    n_rejected = len(tokenizer(row["rejected"]).input_ids)
    print(f"\n────── Example {i + 1} ──────")
    print(f"PROMPT ({n_prompt} tok):\n{textwrap.shorten(row['prompt'], 200)}")
    print(f"\nCHOSEN ({n_chosen} tok):\n{textwrap.shorten(row['chosen'], 250)}")
    print(f"\nREJECTED ({n_rejected} tok):\n{textwrap.shorten(row['rejected'], 250)}")
    assert row["chosen"] != row["rejected"], "chosen == rejected — dataset is corrupt!"

### 3b. Length distribution check

Pairs longer than `MAX_LEN` will be truncated by the trainer. If too many are
clipped, DPO loses signal. Aim for ≥ 80% of pairs fitting.

In [ ]:
import numpy as np

prompt_lens = np.array([len(tokenizer(p).input_ids) for p in pref["prompt"]])
chosen_lens = np.array([len(tokenizer(c).input_ids) for c in pref["chosen"]])
rejected_lens = np.array([len(tokenizer(r).input_ids) for r in pref["rejected"]])

total_len = prompt_lens + np.maximum(chosen_lens, rejected_lens)
fit_pct = (total_len <= MAX_LEN).mean() * 100

print(f"Prompt:   median={np.median(prompt_lens):.0f}  P95={np.percentile(prompt_lens, 95):.0f}")
print(f"Chosen:   median={np.median(chosen_lens):.0f}  P95={np.percentile(chosen_lens, 95):.0f}")
print(f"Rejected: median={np.median(rejected_lens):.0f}  P95={np.percentile(rejected_lens, 95):.0f}")
print(f"\n{fit_pct:.1f}% of pairs fit in MAX_LEN={MAX_LEN}")
if fit_pct < 80:
    print("⚠  Less than 80% fit. Consider increasing MAX_LEN or filtering long pairs.")

## 4. Save Parquet

In [ ]:
pref.to_parquet(str(PREF_OUT / "train.parquet"))
print(f"Saved {len(pref)} pairs to {PREF_OUT / 'train.parquet'}")

# Also save a small eval slice (last 50 pairs) for NB4 use.
eval_slice = pref.select(range(len(pref) - 50, len(pref)))
eval_slice.to_parquet(str(PREF_OUT / "eval.parquet"))
print(f"Saved 50 eval pairs to {PREF_OUT / 'eval.parquet'}")

## 5. Vibe-coding callout

Bạn vừa load 2k cặp English UltraFeedback. Cho VN-aligned model thực sự bạn cần
preference data tiếng Việt. Có 3 con đường (deck §5.3 — `BONUS-CHALLENGE.md`
provocation #1 nếu muốn full):

1. **Translate**: chạy NLLB-3.3B trên 2k cặp này. Quality OK, không native.
2. **Generate native**: 200 prompts VN từ VMLU stems → 2 responses (Lab21-SFT vs
   stronger model như Gemini Flash) → judge với GPT-4o → train DPO trên đó.
3. **Hybrid**: 1.8k UltraFeedback + 200 native VN. Best-of-both.

Notebook 03 dùng English baseline (option 0) cho fairness với deck demo. Nếu
bạn ambitious: thay `data/pref/train.parquet` ở NB3 bằng dataset của bạn — code
sau đó không đổi.

**Next:** NB3 — train DPO trainer với reward curves.

---
# ⏵ Stage from `notebooks/03_dpo_train.py`
---

# NB3 — DPO Training (the main event)

**Stack:** TRL `DPOTrainer` + `DPOConfig(beta=0.1, lr=5e-7)` from deck §5.2.
Maps to deck §3 (DPO derivation), §3.4 (failure modes — read closely!), §5.2 (TRL impl).

> **Mục tiêu:** train DPO adapter on top of NB1 SFT-mini. Plot reward curves
> (cả `chosen_rewards` và `rejected_rewards`). Save adapter to `adapters/dpo/`.
>
> Đây là **the** notebook quan trọng nhất của lab — 25/100 pts đến từ đây.
> Đặc biệt là: **plot cả 2 curve riêng biệt**, không chỉ reward gap (deck §3.4).

## 0. Setup

In [ ]:
import os
from pathlib import Path

COMPUTE_TIER = os.environ.get("COMPUTE_TIER", "T4").upper()

if COMPUTE_TIER == "T4":
    BASE_MODEL = "unsloth/Qwen2.5-3B-bnb-4bit"
    MAX_LEN = 512
    MAX_PROMPT_LEN = 256
    PER_DEVICE_BATCH = 1
    GRAD_ACCUM = 8
else:
    BASE_MODEL = "unsloth/Qwen2.5-7B-bnb-4bit"
    MAX_LEN = 1024
    MAX_PROMPT_LEN = 512
    PER_DEVICE_BATCH = 1
    GRAD_ACCUM = 4

# Hyperparameters from deck §5.2 lines 849–886
BETA = float(os.environ.get("DPO_BETA", "0.1"))
LR = float(os.environ.get("DPO_LR", "5e-7"))
EPOCHS = int(os.environ.get("DPO_EPOCHS", "1"))

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SFT_PATH = REPO_ROOT / "adapters" / "sft-mini"
DPO_OUT = REPO_ROOT / "adapters" / "dpo"
PREF_PATH = REPO_ROOT / "data" / "pref" / "train.parquet"

DPO_OUT.mkdir(parents=True, exist_ok=True)

assert SFT_PATH.exists(), f"NB1 must run first — {SFT_PATH} missing"
assert PREF_PATH.exists(), f"NB2 must run first — {PREF_PATH} missing"

print(f"COMPUTE_TIER:    {COMPUTE_TIER}")
print(f"BASE_MODEL:      {BASE_MODEL}")
print(f"DPO hyperparams: beta={BETA}  lr={LR}  epochs={EPOCHS}")
print(f"max_length:      {MAX_LEN}  (prompt={MAX_PROMPT_LEN})")
print(f"effective batch: {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"SFT input:       {SFT_PATH}")
print(f"output:          {DPO_OUT}")

In [ ]:
import torch

assert torch.cuda.is_available(), "DPO needs a CUDA GPU. See HARDWARE-GUIDE.md."

## 1. Load policy + reference (the VRAM-doubling part)

**Critical:** DPO needs the policy (trainable) AND a frozen reference (no grad).
The reference is the SFT model at step 0; we load it twice. Unsloth's 4-bit base
is shared across copies — only the LoRA adapter differs.

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel

# Policy — gets new DPO LoRA adapter on top of SFT LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_LEN,
    dtype=None,
    load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load SFT adapter on top of base
model = PeftModel.from_pretrained(model, str(SFT_PATH), is_trainable=True)
print(f"Policy: {model.__class__.__name__} with SFT adapter loaded")

In [ ]:
# Wrap policy with NEW LoRA adapter for DPO updates (don't merge SFT — keep stacked)
# Unsloth re-applies LoRA on top of the existing PeftModel.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
print(f"Trainable params (DPO LoRA): {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

> **Why no separate `ref_model=` argument?** Modern TRL (≥ 0.12) auto-detects
> PEFT models and uses the *base model without the adapter* as the reference.
> That's the same memory layout: 1 base + 2 adapter sets in VRAM. No deepcopy
> needed.

## 2. Build DPOConfig (deck §5.2 hyperparameters)

In [ ]:
from trl import DPOConfig

dpo_config = DPOConfig(
    output_dir=str(DPO_OUT.parent / "dpo-checkpoints"),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    beta=BETA,
    max_length=MAX_LEN,
    max_prompt_length=MAX_PROMPT_LEN,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="no",
    optim="adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    seed=42,
    loss_type="sigmoid",         # DPO standard (alternatives: ipo, hinge, kto)
    report_to="none",
)

print(f"DPOConfig: beta={dpo_config.beta}  lr={dpo_config.learning_rate}  loss_type={dpo_config.loss_type}")

## 3. Load preference data

In [ ]:
from datasets import Dataset

pref_ds = Dataset.from_parquet(str(PREF_PATH))
print(f"Loaded {len(pref_ds)} preference pairs from {PREF_PATH}")
print(f"Columns: {pref_ds.column_names}")

## 4. Train

In [ ]:
from trl import DPOTrainer

trainer = DPOTrainer(
    model=model,
    ref_model=None,                # auto-derived from PEFT base
    args=dpo_config,
    train_dataset=pref_ds,
    tokenizer=tokenizer,
)

In [ ]:
train_result = trainer.train()
print(f"\nFinal DPO loss: {train_result.training_loss:.4f}")

## 5. Plot reward curves — THE diagnostic

**Read deck §3.4 before interpreting these.** A growing reward gap can come from:
- **(intended)** chosen reward going up + rejected staying flat
- **(intended)** chosen rising slowly + rejected falling fast
- **(likelihood displacement)** chosen reward going *down* + rejected falling faster

The third case is what Razin et al. 2024 documented. It's not a bug, but it
tells you the model is finding a way to widen the gap that doesn't necessarily
improve actual chosen probability.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

logs = pd.DataFrame(trainer.state.log_history)
logs = logs[logs["loss"].notna() if "loss" in logs.columns else logs.index].copy()

# TRL DPO logs include rewards/chosen, rewards/rejected, rewards/margins, kl
chosen_col = "rewards/chosen" if "rewards/chosen" in logs.columns else None
rejected_col = "rewards/rejected" if "rewards/rejected" in logs.columns else None

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

if chosen_col and rejected_col:
    axes[0].plot(logs["step"], logs[chosen_col], label="chosen reward", color="#2e548a", linewidth=1.5)
    axes[0].plot(logs["step"], logs[rejected_col], label="rejected reward", color="#c83538", linewidth=1.5)
    axes[0].axhline(0, color="#888", linestyle=":", linewidth=0.7)
    axes[0].set_xlabel("Training step")
    axes[0].set_ylabel("Implicit reward (log π/π_ref)")
    axes[0].set_title("Chosen vs Rejected rewards")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    gap = logs[chosen_col] - logs[rejected_col]
    axes[1].plot(logs["step"], gap, color="#1a3355", linewidth=1.8)
    axes[1].axhline(0, color="#888", linestyle=":", linewidth=0.7)
    axes[1].set_xlabel("Training step")
    axes[1].set_ylabel("Reward gap (chosen − rejected)")
    axes[1].set_title("Reward gap (the headline number)")
    axes[1].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, "No reward columns in trainer.state.log_history.\nLikely TRL version mismatch.",
                 ha="center", va="center", transform=axes[0].transAxes)
    axes[1].text(0.5, 0.5, "—", ha="center", va="center", transform=axes[1].transAxes)

fig.suptitle(f"DPO reward curves · {COMPUTE_TIER} · β={BETA} · lr={LR}", y=1.02)
fig.tight_layout()

screenshot_dir = REPO_ROOT / "submission" / "screenshots"
screenshot_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(screenshot_dir / "03-dpo-reward-curves.png", dpi=120, bbox_inches="tight")
plt.show()

### 5a. Failure-mode self-check

Read this cell carefully — it tells you which kind of "reward gap up" you got.

In [ ]:
if chosen_col and rejected_col and len(logs) >= 5:
    last_chosen = logs[chosen_col].iloc[-5:].mean()
    last_rejected = logs[rejected_col].iloc[-5:].mean()
    last_gap = last_chosen - last_rejected
    first_chosen = logs[chosen_col].iloc[:5].mean()

    chosen_delta = last_chosen - first_chosen

    print(f"END  chosen reward:    {last_chosen:+.3f}")
    print(f"END  rejected reward:  {last_rejected:+.3f}")
    print(f"END  reward gap:       {last_gap:+.3f}")
    print()

    if last_gap < 0:
        print("✗ FAILURE: reward gap went NEGATIVE. DPO did the opposite of what you wanted.")
        print("  Likely causes: data quality (chosen/rejected swapped?), beta too high, lr too low.")
    elif chosen_delta < -0.5 and last_gap > 0:
        print("⚠  LIKELIHOOD DISPLACEMENT (deck §3.4):")
        print(f"   Reward gap is positive ({last_gap:+.3f}) — good!")
        print(f"   But chosen reward FELL by {chosen_delta:+.3f} during training.")
        print("   The gap grew because rejected fell faster than chosen.")
        print("   Document this in REFLECTION § 3 — it's a teachable moment, not a bug.")
    elif chosen_delta > 0 and last_gap > 0:
        print("✓ INTENDED: chosen reward UP and gap positive. Classic DPO success.")
    else:
        print("?  AMBIGUOUS: weak chosen movement + positive gap. Try longer training or higher lr.")

## 6. Save adapter

In [ ]:
trainer.model.save_pretrained(str(DPO_OUT))
tokenizer.save_pretrained(str(DPO_OUT))
print(f"Saved DPO adapter to {DPO_OUT}")

# Save the headline metrics for verify.py + REFLECTION
import json

metrics = {
    "compute_tier": COMPUTE_TIER,
    "base_model": BASE_MODEL,
    "beta": BETA,
    "lr": LR,
    "epochs": EPOCHS,
    "final_train_loss": float(train_result.training_loss),
    "end_chosen_reward": float(last_chosen) if chosen_col else None,
    "end_rejected_reward": float(last_rejected) if rejected_col else None,
    "end_reward_gap": float(last_gap) if chosen_col and rejected_col else None,
}
(DPO_OUT / "dpo_metrics.json").write_text(json.dumps(metrics, indent=2))
print(f"Wrote metrics to {DPO_OUT / 'dpo_metrics.json'}")

## 7. Vibe-coding callout

Now's the time for the **β experiment** if you want the +6 rigor add-on.

`make beta-sweep` runs this notebook 3 times with `DPO_BETA ∈ {0.05, 0.1, 0.5}`
and saves to `adapters/dpo-b{0.05,0.1,0.5}/`. Plot the results yourself:

```python
import json
import matplotlib.pyplot as plt
from pathlib import Path

results = []
for d in sorted((REPO_ROOT / "adapters").glob("dpo-b*")):
    m = json.loads((d / "dpo_metrics.json").read_text())
    results.append((m["beta"], m["end_reward_gap"]))
# plot β vs reward_gap
```

**Think-hard zone:** what's the *expected* shape of the β-vs-reward-gap curve?
Hypothesize before you look at the data. (Hint: deck §3.3.)

**Next:** NB4 — qualitative side-by-side comparison.

---
## Part 1 screenshot checklist

Before downloading the zip, capture these screenshots:

1. `01-setup-gpu.png` — after the GPU probe cell near the top of this notebook
2. `02-sft-loss.png` — after the NB1 loss-curve plot cell
3. `03-dpo-reward-curves.png` — after the NB3 reward-curves plot cell
---


---
## Final — Package Part 1 artifacts and auto-download

Run this final cell after NB3 finishes. It creates `day22-colab-part1.zip` with the
handoff artifacts required by Part 2 and triggers a browser download.
---


In [ ]:
from pathlib import Path
from google.colab import files
import json
import zipfile

ROOT = Path('/content/lab22').resolve()
ARCHIVE = ROOT / 'day22-colab-part1.zip'
MANIFEST = ROOT / 'submission' / 'colab_artifact_manifest_part1.json'

REQUIRED_PATHS = [
    'adapters/sft-mini/adapter_config.json',
    'adapters/sft-mini/adapter_model.safetensors',
    'adapters/dpo/adapter_config.json',
    'adapters/dpo/adapter_model.safetensors',
    'adapters/dpo/dpo_metrics.json',
    'data/pref/train.parquet',
    'data/pref/eval.parquet',
]
INCLUDE_GLOBS = [
    'adapters/sft-mini/**',
    'adapters/dpo/**',
    'data/pref/*.parquet',
    'submission/screenshots/*.png',
    'submission/screenshots/*.jpg',
    'submission/screenshots/*.jpeg',
]

missing = [rel for rel in REQUIRED_PATHS if not (ROOT / rel).exists()]
assert not missing, f'Missing Part 1 handoff artifacts: {missing}'

files_to_zip = []
seen = set()
for pattern in INCLUDE_GLOBS:
    for path in sorted(ROOT.glob(pattern)):
        if path.is_file() and path not in seen:
            seen.add(path)
            files_to_zip.append(path)
        elif path.is_dir():
            for child in sorted(path.rglob('*')):
                if child.is_file() and child not in seen:
                    seen.add(child)
                    files_to_zip.append(child)

manifest = {
    'part': 'part1',
    'archive': ARCHIVE.name,
    'required_paths': REQUIRED_PATHS,
    'manual_screenshots_to_take': [
        '01-setup-gpu.png',
        '02-sft-loss.png',
        '03-dpo-reward-curves.png',
    ],
    'packaged_files': [str(path.relative_to(ROOT)) for path in files_to_zip],
}
MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

if ARCHIVE.exists():
    ARCHIVE.unlink()
with zipfile.ZipFile(ARCHIVE, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
    for path in files_to_zip:
        archive.write(path, path.relative_to(ROOT))
    archive.write(MANIFEST, MANIFEST.relative_to(ROOT))

print(f'Created: {ARCHIVE}')
print(f'Size: {ARCHIVE.stat().st_size / 1_000_000:.1f} MB')
print('Download should start automatically...')
files.download(str(ARCHIVE))
